In [ ]:
import os
import json
import random
import hashlib
import sqlite3
from typing import List, Dict, Set, Tuple, Any
from openai import OpenAI
from dotenv import load_dotenv
from tqdm.auto import tqdm

load_dotenv()

# Config

MODEL_NAME = "gpt-4o-mini"
TOTAL_SAMPLES = 3000
BATCH_SIZE = 20
MAX_RETRIES = 3

TRAIN_RATIO = 0.7
VAL_RATIO = 0.15
TEST_RATIO = 0.15

PRICE_PER_1K_INPUT = 0.000150
PRICE_PER_1K_OUTPUT = 0.000600

OUTPUT_DIR = "dataset_output"
CACHE_FILE = "prompt_cache.db"
TMP_JSONL = os.path.join(OUTPUT_DIR, "_tmp_dataset.jsonl")

os.makedirs(OUTPUT_DIR, exist_ok=True)
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))


# SQLite-backed prompt cache and per-complexity checkpoint store

def get_db():
    conn = sqlite3.connect(CACHE_FILE)
    conn.execute("CREATE TABLE IF NOT EXISTS cache (key TEXT PRIMARY KEY, value TEXT)")
    conn.execute("CREATE TABLE IF NOT EXISTS checkpoint (complexity TEXT PRIMARY KEY, round_num INTEGER)")
    conn.commit()
    return conn

def cache_get(key: str):
    with get_db() as conn:
        row = conn.execute("SELECT value FROM cache WHERE key=?", (key,)).fetchone()
        return json.loads(row[0]) if row else None

def cache_set(key: str, value):
    with get_db() as conn:
        conn.execute("INSERT OR REPLACE INTO cache(key, value) VALUES (?,?)", (key, json.dumps(value)))
        conn.commit()

def checkpoint_get(complexity: str) -> int:
    with get_db() as conn:
        row = conn.execute("SELECT round_num FROM checkpoint WHERE complexity=?", (complexity,)).fetchone()
        return row[0] if row else 0

def checkpoint_set(complexity: str, round_num: int):
    with get_db() as conn:
        conn.execute("INSERT OR REPLACE INTO checkpoint(complexity, round_num) VALUES (?,?)", (complexity, round_num))
        conn.commit()

def checkpoint_clear(complexity: str):
    with get_db() as conn:
        conn.execute("DELETE FROM checkpoint WHERE complexity=?", (complexity,))
        conn.commit()

def hash_prompt(prompt: str) -> str:
    return hashlib.sha256(prompt.encode()).hexdigest()


# Schema and complexity definitions

with open("schema_prompt.txt") as f:
    SCHEMA = f.read()

COMPLEXITIES = ["easy", "medium", "hard", "very_hard"]

COMPLEXITY_RULES = {
    "easy":      lambda sql: "JOIN" not in sql.upper(),
    "medium":    lambda sql: sql.upper().count("JOIN") == 1,
    "hard":      lambda sql: 2 <= sql.upper().count("JOIN") <= 3,
    "very_hard": lambda sql: sql.upper().count("JOIN") >= 3 or "SELECT" in sql.upper().split("WHERE")[-1],
}

COMPLEXITY_INSTRUCTIONS = {
    "easy":      "Single table query. Do NOT use any JOINs.",
    "medium":    "Join exactly 2 tables.",
    "hard":      "Join 3-4 tables with aggregation or HAVING.",
    "very_hard": "Multiple joins with nested subqueries and aggregation.",
}


# Restore progress from a previous run

def load_existing_progress() -> Tuple[Set[str], Dict[str, int]]:
    seen_hashes: Set[str] = set()
    counts: Dict[str, int] = {c: 0 for c in COMPLEXITIES}

    if not os.path.exists(TMP_JSONL):
        return seen_hashes, counts

    with open(TMP_JSONL) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                sample = json.loads(line)
                sql = sample.get("sql_query", "")
                sql_hash = hashlib.md5(sql.encode()).hexdigest()
                seen_hashes.add(sql_hash)
                complexity = sample.get("complexity", "")
                if complexity in counts:
                    counts[complexity] += 1
            except json.JSONDecodeError:
                pass

    total = sum(counts.values())
    if total > 0:
        tqdm.write(f"Resuming from checkpoint: {total} samples already written {dict(counts)}")

    return seen_hashes, counts


# Dataset generation

def build_prompt(complexity: str, batch_size: int, round_num: int = 0) -> str:
    variety_hint = f"\nGenerate variety batch #{round_num}." if round_num > 0 else ""
    return f"""Use this PostgreSQL schema:
{SCHEMA}

Generate {batch_size} DIVERSE examples as JSON with key "data":{variety_hint}
{{
  "data": [
    {{"natural_language": "...", "sql_query": "...", "complexity": "{complexity}"}}
  ]
}}

Complexity rule: {COMPLEXITY_INSTRUCTIONS[complexity]}

IMPORTANT: Use PostgreSQL syntax only.
- Use EXTRACT(YEAR FROM x) not YEAR(x)
- Use CURRENT_DATE not CURDATE()
- Use COALESCE(a, b) not IFNULL(a, b)
- Use STRING_AGG(x, ',') not GROUP_CONCAT(x)
- Use INTERVAL '1 year' not INTERVAL 1 YEAR
- Do not use column aliases in HAVING; use the full expression instead"""


def generate_batch(complexity: str, batch_size: int, round_num: int = 0, pbar: tqdm = None) -> Tuple[List[Dict], Any]:
    prompt = build_prompt(complexity, batch_size, round_num)
    prompt_hash = hash_prompt(prompt)

    cached = cache_get(prompt_hash)
    if cached:
        return cached, None

    for attempt in range(MAX_RETRIES):
        try:
            if pbar:
                pbar.set_postfix({"status": "calling API...", "round": round_num}, refresh=True)
            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {"role": "system", "content": "You are a PostgreSQL expert. Generate queries that are valid PostgreSQL syntax only. Always return valid JSON."},
                    {"role": "user", "content": prompt}
                ],
                response_format={"type": "json_object"},
                temperature=0.9
            )
            raw = json.loads(response.choices[0].message.content)
            if isinstance(raw, dict):
                data = raw.get("data") or next((v for v in raw.values() if isinstance(v, list)), [])
            else:
                data = raw

            if not isinstance(data, list):
                raise ValueError("Extracted data is not a list")

            cache_set(prompt_hash, data)
            return data, response.usage

        except Exception as e:
            if pbar:
                pbar.set_postfix({"status": f"retry {attempt+1}/{MAX_RETRIES}"}, refresh=True)
            if attempt == MAX_RETRIES - 1:
                tqdm.write(f"Batch failed after {MAX_RETRIES} attempts: {e}")
                return [], None

    return [], None


def generate_dataset_to_file():
    seen_hashes, existing_counts = load_existing_progress()
    total_input_tokens = total_output_tokens = 0
    per_complexity = TOTAL_SAMPLES // len(COMPLEXITIES)
    total_written = sum(existing_counts.values())

    outer = tqdm(COMPLEXITIES, desc="Overall", unit="complexity")

    with open(TMP_JSONL, "a") as f:
        for complexity in outer:
            outer.set_description(f"Complexity: {complexity}")
            count = existing_counts[complexity]

            if count >= per_complexity:
                tqdm.write(f"  {complexity}: already complete ({count}/{per_complexity}), skipping.")
                checkpoint_clear(complexity)
                continue

            round_num = checkpoint_get(complexity)
            if round_num > 0:
                tqdm.write(f"  {complexity}: resuming from round {round_num} ({count}/{per_complexity} samples)")

            inner = tqdm(total=per_complexity, initial=count, desc=f"  {complexity}", unit="sample", leave=False)

            while count < per_complexity:
                needed = per_complexity - count
                batch, usage = generate_batch(
                    complexity, min(BATCH_SIZE, needed + 5),
                    round_num=round_num, pbar=inner
                )
                round_num += 1
                checkpoint_set(complexity, round_num)

                if usage:
                    total_input_tokens += usage.prompt_tokens
                    total_output_tokens += usage.completion_tokens

                added = 0
                for sample in batch:
                    if not isinstance(sample, dict) or "sql_query" not in sample:
                        continue
                    sql = sample["sql_query"]
                    sql_hash = hashlib.md5(sql.encode()).hexdigest()
                    if sql_hash in seen_hashes or not COMPLEXITY_RULES[complexity](sql):
                        continue
                    seen_hashes.add(sql_hash)
                    sample["complexity"] = complexity
                    f.write(json.dumps(sample) + "\n")
                    f.flush()
                    count += 1
                    total_written += 1
                    added += 1
                    inner.update(1)
                    if count >= per_complexity:
                        break

                if added == 0:
                    inner.set_postfix({"status": f"no new samples, retrying (round {round_num})"}, refresh=True)

            inner.close()
            checkpoint_clear(complexity)

    cost = (total_input_tokens / 1000 * PRICE_PER_1K_INPUT) + (total_output_tokens / 1000 * PRICE_PER_1K_OUTPUT)
    if total_input_tokens > 0:
        tqdm.write(f"\nToken Usage. Input: {total_input_tokens} | Output: {total_output_tokens}")
        tqdm.write(f"Estimated cost: ${cost:.4f}")
    return total_written


# Split the generated samples into train/validation/test files

def split_and_save_jsonl(total: int):
    train_end = int(total * TRAIN_RATIO)
    val_end = train_end + int(total * VAL_RATIO)

    splits = {
        "train.json":      (0, train_end),
        "validation.json": (train_end, val_end),
        "test.json":       (val_end, total),
    }

    with open(TMP_JSONL) as f:
        lines = f.readlines()
    random.seed(42)
    random.shuffle(lines)

    for filename, (start, end) in splits.items():
        out = os.path.join(OUTPUT_DIR, filename)
        with open(out, "w") as f:
            f.write("[\n")
            chunk = lines[start:end]
            for i, line in enumerate(chunk):
                f.write(line.rstrip())
                if i < len(chunk) - 1:
                    f.write(",\n")
            f.write("\n]\n")
        tqdm.write(f"Saved {filename} ({end - start} samples)")


# Run generation and split

tqdm.write("Starting generation...")
total = generate_dataset_to_file()
tqdm.write(f"Total samples generated: {total}")

split_and_save_jsonl(total)
os.remove(TMP_JSONL)
tqdm.write("Done.")